# EGH444 Assessment 2 — Group workflow

Task 1 runs **both RGB baselines through one script** (`Task_1/rgb_segmentation.py`) so they share an identical protocol:

| Setting | Value (both models) |
|---|---|
| Splits | `train.txt` shuffled with seed 42 → 636 train / 159 val; `test.txt` → 654 test |
| Labels | raw IDs 1–40 → 0–39; raw 0 (unlabelled) ignored |
| Input size | 120 × 160 |
| Backbone | ImageNet ResNet50 (dilated, output stride 8), only `layer4` fine-tuned, backbone BN stats frozen |
| Optimiser | AdamW, head LR 1e-3, backbone LR 1e-4, poly decay, ≤12 epochs, early stop (patience 4), batch 2 |
| Model selection | best **val** mIoU; test evaluated only on the selected checkpoint |

The only variable is the head: **FCN** (single 3×3 conv) vs **DeepLabV3** (ASPP multi-scale context).

## Setup

In [ ]:
%pip -q install gdown

import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import gdown
import pandas as pd
from IPython.display import display

In [ ]:
# Clone the repo, or update it if it's already here.
# If the repo is ever made private again, save a GitHub token as a Colab secret named GITHUB_TOKEN.
REPO = Path("/content/EGH444-Assessment2")
BRANCH = "task1-unified-baselines"   # switch to "master" once this is merged
try:
    from google.colab import userdata
    URL = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/cal272b/EGH444-Assessment2.git"
except Exception:
    URL = "https://github.com/cal272b/EGH444-Assessment2.git"


def git(*args):
    subprocess.run(["git", *args], check=True)   # raises instead of failing silently


if REPO.exists() and not (REPO / ".git").exists():
    shutil.rmtree(REPO)                            # leftover from a failed clone
if REPO.exists():
    git("-C", REPO, "remote", "set-url", "origin", URL)
    git("-C", REPO, "fetch", "-q", "origin", BRANCH)
    git("-C", REPO, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}")
else:
    git("clone", "-q", "-b", BRANCH, URL, REPO)
print("Code at", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True).stdout)

TASK_1 = REPO / "Task_1"
SCRIPT = TASK_1 / "rgb_segmentation.py"
RUNS = TASK_1 / "runs"
sys.path.insert(0, str(TASK_1))


def run(*args):
    """Run rgb_segmentation.py and stream its output into the cell."""
    cmd = [sys.executable, str(SCRIPT), *map(str, args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"Command failed ({p.returncode})")

## Dataset

In [ ]:
DATA_ROOT = Path("/content/datasets/NYUDepthv2")

if not (DATA_ROOT / "train.txt").exists():
    zip_path = "/content/NYUDepthv2.zip"
    gdown.download(id="1nGsWNF7zU0O6RZAG-aQIs4HYnxSwP-mr", output=zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall("/content/datasets")
    Path(zip_path).unlink(missing_ok=True)
    DATA_ROOT = next(
        p.parent for p in Path("/content/datasets").rglob("train.txt")
        if (p.parent / "RGB").is_dir() and (p.parent / "Label").is_dir()
    )
print("Dataset:", DATA_ROOT)

## Task 1 — Checkpoints

Each trained model lives at `runs/<run>/<model>/best.pt`, with that run's `splits.json` two levels up
(the script refuses to evaluate a checkpoint on different sample IDs).

In [ ]:
# Back up / restore trained runs on Google Drive so a Colab reset doesn't lose them
USE_DRIVE = True
DRIVE_RUNS = Path("/content/drive/MyDrive/EGH444/runs")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RUNS.mkdir(parents=True, exist_ok=True)


def restore(run_name):
    src = DRIVE_RUNS / run_name
    if USE_DRIVE and src.exists() and not (RUNS / run_name).exists():
        shutil.copytree(src, RUNS / run_name)
        print("Restored", run_name, "from Drive")


def backup(run_name):
    if USE_DRIVE:
        shutil.copytree(RUNS / run_name, DRIVE_RUNS / run_name, dirs_exist_ok=True)
        print("Backed up", run_name, "to Drive")

In [ ]:
# DeepLabV3 (Dylan) — existing trained weights + the original run's splits/config
DEEPLAB_RUN = "deeplab-full-corrected"
DEEPLAB_CKPT = RUNS / DEEPLAB_RUN / "deeplabv3/best.pt"
DEEPLAB_CKPT.parent.mkdir(parents=True, exist_ok=True)

if not DEEPLAB_CKPT.exists():
    gdown.download(id="1nLs0MFeoaaIBUxXlq7iwrF-pexfdW820", output=str(DEEPLAB_CKPT))
for filename in ["splits.json", "config.json"]:
    shutil.copy2(TASK_1 / "DeepLab_Results" / DEEPLAB_RUN / filename, RUNS / DEEPLAB_RUN / filename)

In [ ]:
# FCN — retrained under the SAME protocol as DeepLabV3, so only the head differs.
# The earlier template FCN run isn't comparable: 41 classes (unlabelled counted as a class),
# 240x320 input, full-backbone training, and its best epoch was picked on the test set.
FCN_RUN = "fcn-baseline"
FCN_CKPT = RUNS / FCN_RUN / "fcn/best.pt"

restore(FCN_RUN)
if not FCN_CKPT.exists():
    run("--data-root", DATA_ROOT, "--model", "fcn",
        "--output-root", RUNS, "--run-name", FCN_RUN, "--device", "auto")
    backup(FCN_RUN)
else:
    print("Using existing", FCN_CKPT)

## Task 1 — Evaluate both on val and test

In [ ]:
CHECKPOINTS = {"DeepLabV3": DEEPLAB_CKPT, "FCN": FCN_CKPT}
INCLUDE_TEST = True   # both checkpoints were selected on val only, so reporting test is legitimate

EVAL = {}
for name, ckpt in CHECKPOINTS.items():
    eval_name = f"{name.lower()}-eval"
    shutil.rmtree(RUNS / eval_name, ignore_errors=True)   # the script won't overwrite a run folder
    args = ["--data-root", DATA_ROOT, "--evaluate-checkpoint", ckpt,
            "--output-root", RUNS, "--run-name", eval_name, "--device", "auto"]
    if not INCLUDE_TEST:
        args.append("--validation-only")
    run(*args)
    EVAL[name] = RUNS / eval_name

## Task 1 — Side-by-side comparison

In [ ]:
import importlib
importlib.invalidate_caches()   # Task_1 may not have existed when sys.path was first scanned
from compare_runs import compare

COMPARISON = TASK_1 / "comparison"
summary = compare(EVAL, out_dir=COMPARISON)
display(pd.DataFrame(summary).set_index(["split", "model"]))

In [ ]:
# Classes where the two heads differ most on val (positive = DeepLabV3 better)
classwise = pd.read_csv(COMPARISON / "classwise_iou_val.csv")
diff_col = next(c for c in classwise if c.startswith("diff_"))
display(classwise.reindex(classwise[diff_col].abs().sort_values(ascending=False).index).head(10))

In [ ]:
# Weakest val images per model — starting point for the Task 2 failure analysis
for name, folder in EVAL.items():
    per_image = pd.read_csv(folder / "val/per_image.csv").sort_values("mIoU")
    print(name)
    display(per_image.head(5))

## Task 2 — RGB improvements

Train each change as a new run, one change at a time against the matching baseline, e.g.
`run("--data-root", DATA_ROOT, "--model", "deeplabv3", "--size", 240, 320, "--output-root", RUNS, "--run-name", "deeplab-240x320")`,
evaluate it as above, add it to `EVAL`, and re-run `compare`. `compare` refuses runs evaluated at different
sizes, because metrics are computed on resized masks; a resolution experiment first needs evaluation at a common
label size.